**OBJETIVOS:**

**1: Web scraping de la pagina web  https://www.argenprop.com/**

**2: Organizar los datos que se necesitan, graficarlos en lo posible**

**3: Correr un algoritmo de regresion lineal sobre el precio y las caracteristicas dadas**

**4: WIP, Crear un algoritmo que identifica los outliers abajo de la curva**

**5: Crear una herramienta que envia notificaciones a los mails una vez encontrado un departamento en buen precio**

In [1]:
#Importo bibliotecas necesarias y utiles
import pandas as pd
import numpy as np
import matplotlib as mp
import matplotlib.pyplot as plt


from bs4 import BeautifulSoup as bs
import re
import itertools

In [2]:
#!pip install httpx
import httpx as ht

In [4]:
HTTP_REQUEST

<Response [301 Moved Permanently]>

In [3]:

URL_TO_FETCH = f'https://www.argenprop.com/departamento-alquiler-sub-barrio-botanico-sub-barrio-palermo-hollywood-sub-barrio-palermo-nuevo-sub-barrio-parque-las-heras-sub-barrio-br-parque-saavedra-barrio-belgrano-barrio-colegiales-barrio-nunez-2-dormitorios-2-ambientes-y-3-ambientes'

HTTP_REQUEST = ht.get(URL_TO_FETCH)

tree = bs(HTTP_REQUEST, 'html.parser')

#Extraemos la cantidad de paginas
cantidad_de_paginas = tree.find('ul', class_='pagination pagination--links')

li_tags = cantidad_de_paginas.find_all('li')
PagsNum = int(((li_tags[-2]).text))


#Extraemos el total ids en todas las paginas
lista_de_ids = []
for pag in range(1, PagsNum + 1):
    if pag != 1 and pag != 0:
        URL_TO_FETCH = f'https://www.argenprop.com/departamento-alquiler-sub-barrio-botanico-sub-barrio-palermo-hollywood-sub-barrio-palermo-nuevo-sub-barrio-parque-las-heras-sub-barrio-br-parque-saavedra-barrio-belgrano-barrio-colegiales-barrio-nunez-2-dormitorios-2-ambientes-y-3-ambientes-pagina-{pag}'
    else:
        URL_TO_FETCH = f'https://www.argenprop.com/departamento-alquiler-sub-barrio-botanico-sub-barrio-palermo-hollywood-sub-barrio-palermo-nuevo-sub-barrio-parque-las-heras-sub-barrio-br-parque-saavedra-barrio-belgrano-barrio-colegiales-barrio-nunez-2-dormitorios-2-ambientes-y-3-ambientes'
        
    HTTP_REQUEST = ht.get(URL_TO_FETCH)
    tree = bs(HTTP_REQUEST, 'html.parser')

    input_where_ids_live = tree.find(id='ga-dimension-list')
    prop_of_input = input_where_ids_live['data-ids-avisos-mostrados']
    lista_de_ids.append(prop_of_input.split(','))

lista_unica = list(itertools.chain.from_iterable(lista_de_ids))


#Aqui iteramos para cada publicacion individual y extraemos la lista de caracteristicas del departamento
data = []
for id in lista_unica:
    URL_TO_FETCH2 = f'https://www.argenprop.com/departamento-alquiler-sub-barrio-botanico-sub-barrio-palermo-hollywood-sub-barrio-palermo-nuevo-sub-barrio-parque-las-heras-sub-barrio-br-parque-saavedra-barrio-belgrano-barrio-colegiales-barrio-nunez-2-dormitorios-2-ambientes-y-3-ambientes--{id}'
    HTTP_REQUEST = ht.get(URL_TO_FETCH2)
    tree = bs(HTTP_REQUEST, 'html.parser')
    
    caracteristicas = tree.find(class_='property-main')
    atributos_de_la_propiedad = caracteristicas.find(class_='property-main-features')
    elementos = atributos_de_la_propiedad.find_all(recursive=False)
    
    
    #costos = tree.find('div', class_='titlebar__price').text 
    '''Como el precio esta por fuera de property-main-features, me traigo a su tag padre y luego separo el precio de los atributos de la propiedad'''
    precio = caracteristicas.find('p', class_='titlebar__price') if caracteristicas else None
    expensa = caracteristicas.find('p', class_='titlebar__expenses') if caracteristicas else None
    Barrio= caracteristicas.find('h2', class_='titlebar__title') if caracteristicas else None
    
    price_parsed = precio.text if precio else None
    expensa_parsed = expensa.text if expensa else None

    precios = {'precio': price_parsed} if price_parsed else {}
    expensas = {'expensas': expensa_parsed} if expensa_parsed else {}
    data_porID = {}
    for elem in elementos:
        titulo = elem.get('title')
        valor = elem.find('p', class_='strong').text.strip()
        data_porID[titulo]=valor
    data.append({**precios, **expensas, **({'Barrio': Barrio.text} if Barrio else {}), **data_porID})



AttributeError: 'NoneType' object has no attribute 'find_all'

In [4]:
df= pd.DataFrame(data, index=lista_unica)
df

,precio,expensas,Barrio,Sup. cubierta,Dormitorios,Antiguedad,Baños,Ambientes,Cocheras,Estado,Disposición,Orientación,Toilettes,Apto profesional,Antigüedad
12931105,\n USD 1.100\n,\n + $70.000 expensas\n,"Departamento en Alquiler en Botanico, Palermo",85 m² Cubierta,2 dormitorios,A Estrenar,3 baños,3 ambientes,1 cocheras,NaN,NaN,NaN,NaN,NaN,NaN
12808714,\n USD 400\n,\n + $20.000 expensas\n,"Departamento en Alquiler en Belgrano, Capital ...",NaN,2 dormitorios,NaN,1 baño,3 ambientes,1 cocheras,Muy Bueno,Lateral,NaN,NaN,NaN,NaN
12832698,\n USD 800\n,\n + $50.000 expensas\n,"Departamento en Alquiler en Belgrano, Capital ...",73 m² Cubierta,2 dormitorios,25 años,2 baños,3 ambientes,1 cocheras,Muy Bueno,Frente,Suroeste,NaN,NaN,NaN
12728634,\n USD 1.000\n,\n + $15.000 expensas\n,"Departamento en Alquiler en Lomas de Nuñez, Nuñez",75 m² Cubierta,2 dormitorios,5 años,1 baño,3 ambientes,NaN,Excelente,Contra Frente,NaN,1 toilettes,Apto profesi.,NaN
12760378,\n USD 1.650\n,NaN,"Departamento en Alquiler en Las Cañitas, Belgrano",67 m² Cubierta,2 dormitorios,10 años,2 baños,3 ambientes,1 cocheras,Excelente,Contra frente,Noroeste,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12940659,\n USD 1.200\n,NaN,"Departamento en Alquiler en Belgrano, Capital ...",65 m² Cubierta,2 dormitorios,40 años,1 baño,3 ambientes,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12955810,\n $ 450.000\n,\n + $43.550 expensas\n,"Departamento en Alquiler en Belgrano, Capital ...",116 m² Cubierta,2 dormitorios,40 años,2 baños,3 ambientes,1 cocheras,NaN,NaN,NaN,NaN,NaN,NaN
12956805,\n $ 160.000\n,\n + $12.600 expensas\n,"Departamento en Alquiler en Colegiales, Capita...",52 m² Cubierta,2 dormitorios,35 años,1 baño,3 ambientes,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12956872,\n USD 1.300\n,\n + $53.000 expensas\n,"Departamento en Alquiler en Belgrano C, Belgrano",75 m² Cubierta,2 dormitorios,10 años,2 baños,3 ambientes,1 cocheras,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
Datos= df.copy()
#Primero trabajo las columnas faciles
Datos['Sup. cubierta'] = df['Sup. cubierta'].str.extract(r'(\d+)').astype(float)
Datos['Baños'] = df['Baños'].str.extract(r'(\d+)').astype(float)
Datos['Dormitorios'] = df['Dormitorios'].str.extract(r'(\d+)').astype(float)
Datos['Ambientes'] = df['Ambientes'].str.extract(r'(\d+)').astype(float)
Datos['Cocheras'] = df['Cocheras'].str.extract(r'(\d+)').astype(float)
Datos['Antiguedad'] = df['Antiguedad'].str.extract(r'(\d+)').astype(float)
Datos['Toilettes'] = df['Toilettes'].str.extract(r'(\d+)').astype(float)
Datos = Datos.drop('Antigüedad', axis=1)

#lleno los NA
Datos['Antiguedad']= Datos['Antiguedad'].fillna(0)
Datos['Sup. cubierta'] = Datos['Sup. cubierta'].fillna(0)
Datos['Baños'] = Datos['Baños'].fillna(0)
Datos['Dormitorios'] = Datos['Dormitorios'].fillna(0)
Datos['Ambientes'] = Datos['Ambientes'].fillna(0)
Datos['Cocheras'] = Datos['Cocheras'].fillna(0)
Datos['Toilettes'] = Datos['Toilettes'].fillna(0)

#Columna Barrio
Datos['Barrio'] = Datos['Barrio'].str.split('en').str[3].str.strip()
Barrios = ['Belgrano', 'Nuñez', 'Palermo','Colegiales']
patron = '|'.join(Barrios)
Datos['Barrio'] = Datos['Barrio'].str.extract(f'({patron})', flags=re.IGNORECASE)


#Ahora trabajo las columnas de precio y expensas
Datos['expensas']= df['expensas'].str.extract(r'(\d+)').astype(float)
Datos['expensas']= Datos['expensas']*1000
Datos['expensas']= Datos['expensas'].fillna(0)

# Limpiar la columna "precio" y extraer los valores numéricos
Datos['precio'] = Datos['precio'].str.strip()  
Datos['Moneda'] = Datos['precio'].str.extract(r'^\s*(\D+)\s*\d')  # Extrae la moneda (USD o $)

Datos['precio'] = Datos['precio'].str.replace('.', '')
Datos['precio'] = Datos['precio'].str.replace('.', '')
Datos['precio'] = Datos['precio'].str.extract(r'(\d+)').astype(float)  # Extrae los valores numéricos

cotizacion_usd=250 #CAMBIAR
Datos['Moneda'] = Datos['Moneda'].str.strip()
Datos.loc[Datos['Moneda'] == 'USD', 'precio'] *= cotizacion_usd

Datos['Total']=Datos['precio']+ Datos['expensas']

Datos

/tmp/ipykernel_20/1089353392.py:37: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  Datos['precio'] = Datos['precio'].str.replace('.', '')
/tmp/ipykernel_20/1089353392.py:38: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  Datos['precio'] = Datos['precio'].str.replace('.', '')


,precio,expensas,Barrio,Sup. cubierta,Dormitorios,Antiguedad,Baños,Ambientes,Cocheras,Estado,Disposición,Orientación,Toilettes,Apto profesional,Moneda,Total
12931105,275000.0,70000.0,Palermo,85.0,2.0,0.0,3.0,3.0,1.0,NaN,NaN,NaN,0.0,NaN,USD,345000.0
12808714,100000.0,20000.0,Belgrano,0.0,2.0,0.0,1.0,3.0,1.0,Muy Bueno,Lateral,NaN,0.0,NaN,USD,120000.0
12832698,200000.0,50000.0,Belgrano,73.0,2.0,25.0,2.0,3.0,1.0,Muy Bueno,Frente,Suroeste,0.0,NaN,USD,250000.0
12728634,250000.0,15000.0,Nuñez,75.0,2.0,5.0,1.0,3.0,0.0,Excelente,Contra Frente,NaN,1.0,Apto profesi.,USD,265000.0
12760378,412500.0,0.0,Belgrano,67.0,2.0,10.0,2.0,3.0,1.0,Excelente,Contra frente,Noroeste,0.0,NaN,USD,412500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12940659,300000.0,0.0,Belgrano,65.0,2.0,40.0,1.0,3.0,0.0,NaN,NaN,NaN,0.0,NaN,USD,300000.0
12955810,450000.0,43000.0,Belgrano,116.0,2.0,40.0,2.0,3.0,1.0,NaN,NaN,NaN,0.0,NaN,$,493000.0
12956805,160000.0,12000.0,Colegiales,52.0,2.0,35.0,1.0,3.0,0.0,NaN,NaN,NaN,0.0,NaN,$,172000.0
12956872,325000.0,53000.0,Belgrano,75.0,2.0,10.0,2.0,3.0,1.0,NaN,NaN,NaN,0.0,NaN,USD,378000.0


In [6]:
cantidad_usd = Datos['Moneda'].value_counts().get('USD', 0)
print("Cantidad de dptos en USD:", cantidad_usd)

Cantidad de dptos en USD: 106


In [7]:
features = ['Dormitorios', 'Baños', 'Ambientes', 'Cocheras','Sup. cubierta','Antiguedad','Toilettes','Barrio','Moneda',]
X = Datos[features]
y = Datos['Total']


In [8]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


In [9]:
#Realizo One Hot Encoding de las variables, BARRIO, MONEDA
X_encoded = pd.get_dummies(X, columns=['Barrio', 'Moneda'])

# Normalizamos los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

model = IsolationForest(contamination=0.1)  # Especifica la proporción de valores atípicos esperados
model.fit(X_scaled)

y_pred = model.predict(X_scaled)

anomaly_labels = (y_pred == -1)

In [10]:
indices = np.where(anomaly_labels)[0]
dptos_raros = Datos.iloc[indices]
dptos_raros = list(dptos_raros.index.values) #Obtengo indices
print("Dtos Anomalos:")
print(dptos_raros)

indices_str = ', '.join(map(str, dptos_raros))

Dtos Anomalos:
['12705067', '12897464', '12694406', '12694413', '10628544', '12042975', '12767916', '12805920', '12809249', '12881020', '12924169', '12956805', '12957947']


In [11]:

import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from kaggle_secrets import UserSecretsClient

# Obtengo el valor secreto
user_secrets = UserSecretsClient()
secret_value = user_secrets.get_secret("PutoelQueLee")

# Configura los detalles del correo electrónico
sender_email = 'aureliogialluca@gmail.com'
receiver_email = ['aureliogialluca@gmail.com', 'gazzani.fran@gmail.com']
subject = 'Resultados'
message = 'Aquí están los resultados del algoritmo de HOY: '+ indices_str

# Crea MIMEText y configura los encabezados
msg = MIMEMultipart()
msg['From'] = sender_email
msg['To'] = ', '.join(receiver_email)
msg['Subject'] = subject
msg.attach(MIMEText(message, 'plain'))

# Conecta al servidor SMTP y envia el correo electrónico
smtp_server = 'smtp.gmail.com'
smtp_port = 587
smtp_username = 'aureliogialluca@gmail.com'
smtp_password = secret_value

try:
    server = smtplib.SMTP(smtp_server, smtp_port)
    server.starttls()
    server.login(smtp_username, smtp_password)
    server.send_message(msg)
    server.quit()
    print('Correo electrónico enviado exitosamente')
except Exception as e:
    print('Error al enviar el correo electrónico:', str(e))


Correo electrónico enviado exitosamente


Mail, Fran: gazzani.fran@gmail.com